# `mini-conductor`: Colab A100 / H100 run

One question: when a worker never seen in training (Claude Haiku) is added at test time, does an RL-trained Conductor use it, and does it route by worker *description* or by *ID slot*?

Training pool = local Qwen2.5-1.5B + Qwen2.5-7B (RL makes no API calls). Run the cells top to bottom. After a disconnect, re-run cells 1-3 and then the cell you were on: checkpoints, caches, SFT data and eval summaries live on Google Drive and finished work is skipped.

In [ ]:
# 1. Drive + GPU (needs an A100 / H100 runtime)
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# 2. Code: clone (or update) the repo, install, unit tests
import os, subprocess
from getpass import getpass
REPO = 'github.com/xiao98/mini-conductor.git'
PROJECT_DIR = '/content/mini-conductor'
EXP_ROOT = '/content/drive/MyDrive/mc_exp_weak'   # checkpoints / caches / SFT data / eval / figures
os.makedirs(EXP_ROOT, exist_ok=True)
if os.path.isdir(f'{PROJECT_DIR}/.git'):
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only'], check=True)
elif subprocess.run(['git', 'clone', '-q', f'https://{REPO}', PROJECT_DIR]).returncode != 0:
    # Private repo: paste a GitHub token with read access (kept in memory only, removed from the remote URL)
    tok = getpass('GitHub token: ')
    subprocess.run(['git', 'clone', '-q', f'https://{tok}@{REPO}', PROJECT_DIR], check=True)
    subprocess.run(['git', '-C', PROJECT_DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'], check=True)
os.chdir(PROJECT_DIR)
!git log --oneline -1
!pip install -q -r requirements.txt
!pytest -q

In [ ]:
# 3. Claude API credentials (memory only; never written to disk)
os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')
base = input('ANTHROPIC_BASE_URL (leave empty for the official API): ').strip()
if base:
    os.environ['ANTHROPIC_BASE_URL'] = base
# ONE minimal request per Claude model referenced by the pool (here: only the held-out Haiku)
!python scripts/00_check_api.py

In [ ]:
# 4. Datasets (deterministic; rebuilt on a fresh runtime): MATH L3-5 / MATH-500 + MBPP + MMLU-Pro STEM
!python scripts/01_build_dataset.py --output_dir data

In [ ]:
# 5. GATE (no training): each pool worker answers directly + a static topology, on the same 150
#    tasks the matrix evaluates (the matrix reuses this summary). API use: 150 Haiku calls.
#    Expected from earlier measurements: qwen-1.5b ~51% < qwen-7b ~74% < claude-haiku ~93%.
!python scripts/05_eval_suite.py --baselines --skip_conductor --limit 150 --output_dir {EXP_ROOT}/eval/baselines --cache_db {EXP_ROOT}/worker_cache.sqlite
import json
for r in json.load(open(f'{EXP_ROOT}/eval/baselines/eval_summary.json'))['results']:
    print(f"{r['label']:24s} acc={r['accuracy']:.3f} +-{r['acc_ci95']:.3f}", {k: round(v['accuracy'], 3) for k, v in r['by_task_type'].items()})

In [ ]:
# 6. Stage 0: warm start. Haiku writes plans (teacher) and stands in for the local workers; only
#    verified plans are kept. This is the only training-side API use (~4-5 Haiku calls per sample).
#    Skipped if the SFT data / adapter already exist on Drive.
SFT_DATA = f'{EXP_ROOT}/sft_warmup.jsonl'
if not os.path.exists(SFT_DATA):
    !python scripts/02_gen_sft_data.py --dataset data/train.jsonl --output {SFT_DATA} --num_samples 150 --teacher_model claude-haiku-5-5 --cache_db {EXP_ROOT}/sft_cache.sqlite
!wc -l {SFT_DATA}
if not os.path.exists(f'{EXP_ROOT}/sft_warmup/adapter_config.json'):
    !python scripts/03_train_sft.py --data {SFT_DATA} --output_dir {EXP_ROOT}/sft_warmup --epochs 2

In [ ]:
# 7. PILOT: 10 GRPO steps of the base arm + a 30-task eval, in a separate root. Read off the real
#    seconds per step and peak GPU memory here before committing to the full runs.
import time
t0 = time.time()
!python scripts/06_run_matrix.py --root {EXP_ROOT}_pilot --init_adapter {EXP_ROOT}/sft_warmup --only base --total_steps 10 --save_every 5 --eval_limit 30 --skip_baselines
print(f'pilot wall-clock: {(time.time() - t0) / 60:.1f} min')
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!tail -n 3 {EXP_ROOT}_pilot/runs/base/train_metrics.jsonl

In [ ]:
# 8. The two arms (fixed IDs vs IDs shuffled per episode): train -> 4-condition eval -> figures.
#    Resumable: re-run this cell after any disconnect.
!python scripts/06_run_matrix.py --root {EXP_ROOT} --init_adapter {EXP_ROOT}/sft_warmup --only base,rq3_perm

In [ ]:
# 9. Results: accuracy per test-time condition + how often each worker was called
import collections, glob, json
from IPython.display import Image, display
for run in ['base', 'rq3_perm']:
    p = f'{EXP_ROOT}/eval/{run}/eval_summary.json'
    if not os.path.exists(p):
        continue
    for r in json.load(open(p))['results']:
        f = f"{EXP_ROOT}/eval/{run}/details_{r['condition'].replace('+', 'plus_')}.json"
        used = collections.Counter(w for d in json.load(open(f)) for w in d['workers_used'])
        tot = sum(used.values()) or 1
        print(f"{run:9s} {r['condition']:14s} acc={r['accuracy']:.3f} +-{r['acc_ci95']:.3f}", {k: round(v / tot, 2) for k, v in used.most_common()})
for f in sorted(glob.glob(f'{EXP_ROOT}/figures/*.png')):
    print(f); display(Image(f))